<a href="https://colab.research.google.com/github/ltittle/QNC/blob/main/QNC_Presentation_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# C is your CaImAn output: shape (n_neurons, n_timepoints)
# Here I'm faking some example data:
# C = np.load('your_C_matrix.npy')

n_neurons, n_time = C.shape
offset = 2.0  # adjust separation between traces

plt.figure(figsize=(12, 8))
time = np.arange(n_time)

for i in range(n_neurons):
    plt.plot(time, C[i] + i * offset, lw=0.7)

plt.xlabel("Time (frames)")
plt.ylabel("Neuron index (offset)")
plt.title("Calcium Trace Waterfall Plot")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. ΔF/F traces from CNMF-E (all neurons)
dff = cnmfe_model.estimates.C  # shape (cells x time)

# 2. Filter to accepted neurons
accepted_neurons = cnmfe_model.estimates.idx_components
dff = dff[accepted_neurons]  # now only valid neurons

# 3. Compute shape and time vector
n_cells, n_time = dff.shape
fr = cnmfe_model.params.data['fr']
time = np.arange(n_time) / fr

# 4. Z-score each neuron (optional)
dff_z = (dff - np.mean(dff, axis=1, keepdims=True)) / np.std(dff, axis=1, keepdims=True)

# 5. Compute offsets for waterfall
trace_range = np.max(np.ptp(dff_z, axis=1))
offsets = np.arange(n_cells)[::-1] * (trace_range * 1.3)

# 6. Plot waterfall
fig, ax = plt.subplots(figsize=(10, 8))
for idx in range(n_cells):
    ax.plot(time, dff_z[idx] + offsets[idx], linewidth=0.8)

# Add neuron numbers on the left
ax.set_yticks(offsets)
ax.set_yticklabels(np.arange(1, n_cells+1))
ax.set_xlabel("Time (s)")
ax.set_ylabel("Neuron #")
ax.set_title("Homecage Activity (Accepted Neurons)")
plt.tight_layout()
plt.show()

# 7. Print std and SNR for each neuron
stds = np.std(dff, axis=1)

# Rough SNR: peak ΔF/F / baseline std (first 50 frames as baseline)
baseline_std = np.std(dff[:, :50], axis=1)
peak = np.max(dff, axis=1) - np.median(dff, axis=1)
snrs = peak / (baseline_std + 1e-10)

for i in range(n_cells):
    print(f"Neuron {i+1}: std = {stds[i]:.4f}, SNR ≈ {snrs[i]:.2f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ΔF/F traces from CNMF-E
dff = cnmfe_model.estimates.C  # shape (cells x time)
n_cells, n_time = dff.shape

# Build time vector
fr = cnmfe_model.params.data['fr']
time = np.arange(n_time) / fr

# Decide spacing between traces based on amplitude
trace_range = np.max(np.ptp(dff, axis=1))
offsets = np.arange(n_cells)[::-1] * (trace_range * 1.3)

# Plot stacked waterfall
fig, ax = plt.subplots(figsize=(10, 8))

for idx in range(n_cells):
    ax.plot(time, dff[idx] + offsets[idx], linewidth=0.8)

ax.set_xlabel("Time (s)")
ax.set_yticks([])
ax.set_title("Waterfall plot of accepted CNMF-E calcium traces")
plt.tight_layout()
plt.show()
